In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import warnings
warnings.filterwarnings("ignore")

# Load models and config
xgb_model = joblib.load("models/xgb_model.pkl")
lgb_model  = joblib.load("models/lgb_model.pkl")

with open("models/model_config.json", "r") as f:
    model_config = json.load(f)

FEATURE_COLS = model_config["feature_cols"]
THRESHOLD    = model_config["threshold"]

# Load feature matrix
df = pd.read_csv("features.csv")
X  = df[FEATURE_COLS].copy()
y  = df["actual_default"].astype(int).copy()

# Load riders table for segment information
riders = pd.read_csv("riders_clean.csv")



In [2]:
# Seasonal calendar — months where income dips are expected
RAIN_MONTHS     = [3, 4, 5, 10, 11]
RAIN_MONTH_NAMES = {
    3: "March", 4: "April", 5: "May",
    10: "October", 11: "November"
}

def run_fairness_checks(rider_features, scoring_result,
                         requested_amount, application_month=None):
    """
    Run all 9 fairness checks for one rider application.

    Returns a fairness report dict with:
      - passed:   list of checks that passed cleanly
      - warnings: list of checks that fired a warning
      - blocks:   list of hard blocks that override the decision
      - adjusted: whether any adjustment was made to the score
    """

    checks_passed  = []
    checks_warnings= []
    checks_blocks  = []
    adjustments    = []

    credit_score   = scoring_result["credit_score"]
    estimated_ndi  = rider_features.get("estimated_ndi", 8000)
    monthly_income = rider_features.get("monthly_income_estimate", 20000)

    # ────────────────────────────────────────────────────────────────────────
    # CHECK 1 — Seasonal Income Adjustment
    # Do not penalise riders for rain season income dips
    # ────────────────────────────────────────────────────────────────────────
    rain_dip  = rider_features.get("rain_season_dip", 0)
    income_cv = rider_features.get("income_volatility_cv", 0)

    if application_month in RAIN_MONTHS:
        checks_warnings.append({
            "check":   "CHECK 1 — Seasonal Income",
            "status":  "  RAIN SEASON APPLICATION",
            "detail":  f"Application received in {RAIN_MONTH_NAMES.get(application_month, 'rain month')}. "
                       f"Income dips during this period are seasonal and consistent "
                       f"with all Nairobi boda-boda riders. "
                       f"Income volatility not penalised this month.",
            "action":  "Income volatility penalty waived for this application"
        })
        adjustments.append("seasonal_adjustment")
    elif rain_dip > 0.20:
        checks_passed.append({
            "check":  "CHECK 1 — Seasonal Income",
            "status": "✅ PASSED",
            "detail": f"Rain season income dip of {rain_dip*100:.0f}% detected "
                      f"and confirmed as seasonal pattern. Not penalised."
        })
    else:
        checks_passed.append({
            "check":  "CHECK 1 — Seasonal Income",
            "status": " PASSED",
            "detail": "No significant rain season dip detected."
        })

    # ────────────────────────────────────────────────────────────────────────
    # CHECK 2 — Income Gap vs Income Loss Distinction
    # Short gaps are recoverable — not the same as declining income
    # ────────────────────────────────────────────────────────────────────────
    max_gap      = rider_features.get("income_gap_max_days", 0)
    income_trend = rider_features.get("income_trend_90d", 0)

    if max_gap <= 5:
        checks_passed.append({
            "check":  "CHECK 2 — Income Gaps",
            "status": " PASSED",
            "detail": f"Longest income gap: {max_gap} days. "
                      f"Short gaps are normal rest or maintenance days."
        })
    elif max_gap <= 14 and income_trend >= 0:
        checks_warnings.append({
            "check":   "CHECK 2 — Income Gaps",
            "status":  " RECOVERY GAP DETECTED",
            "detail":  f"Income gap of {max_gap} days detected but "
                       f"income recovered afterward. "
                       f"Classified as temporary disruption, not income loss.",
            "action":  "Gap penalty reduced by 50%"
        })
        adjustments.append("gap_penalty_reduced")
    else:
        checks_warnings.append({
            "check":   "CHECK 2 — Income Gaps",
            "status":  "⚠️  EXTENDED GAP",
            "detail":  f"Income gap of {max_gap} days detected. "
                       f"Manual review recommended to understand cause.",
            "action":  "Flagged for loan officer review"
        })

    # ────────────────────────────────────────────────────────────────────────
    # CHECK 3 — Gender Exclusion Confirmation
    # Gender must never be used in scoring
    # ────────────────────────────────────────────────────────────────────────
    checks_passed.append({
        "check":  "CHECK 3 — Gender Exclusion",
        "status": "CONFIRMED",
        "detail": "Gender was not used in any part of the credit score calculation. "
                  "Compliant with Kenya Data Protection Act 2019."
    })

    # ────────────────────────────────────────────────────────────────────────
    # CHECK 4 — Data Richness Assessment
    # Thin data reduces confidence — does not automatically reduce score
    # ────────────────────────────────────────────────────────────────────────
    richness_score = 0

    if rider_features.get("total_income_90d", 0) > 0:
        richness_score += 3    # M-Pesa data present
    if rider_features.get("sacco_tenure_months", 0) > 0:
        richness_score += 3    # SACCO records present
    if rider_features.get("has_app", 0) == 1:
        richness_score += 2    # App trip data present
    if rider_features.get("total_loans_taken", 0) > 0:
        richness_score += 2    # Loan history present

    # Maximum richness score = 10
    if richness_score >= 7:
        checks_passed.append({
            "check":  "CHECK 4 — Data Richness",
            "status": f"ADEQUATE ({richness_score}/10)",
            "detail": "Sufficient data sources present. Score confidence is HIGH."
        })
    elif richness_score >= 4:
        checks_warnings.append({
            "check":   "CHECK 4 — Data Richness",
            "status":  f"  THIN DATA ({richness_score}/10)",
            "detail":  "Limited data sources available. Score confidence is MEDIUM. "
                       "Score has not been reduced — uncertainty noted.",
            "action":  "Maximum first loan capped at KES 10,000"
        })
        adjustments.append("thin_data_cap")
    else:
        checks_warnings.append({
            "check":   "CHECK 4 — Data Richness",
            "status":  f"  VERY THIN DATA ({richness_score}/10)",
            "detail":  "Very limited data. Score based on minimal evidence.",
            "action":  "Maximum loan capped at KES 5,000. Enhanced manual review required."
        })
        adjustments.append("very_thin_data_cap")

    # ────────────────────────────────────────────────────────────────────────
    # CHECK 5 — First Loan Pathway
    # First time borrowers are unknown — not risky
    # ────────────────────────────────────────────────────────────────────────
    first_timer = rider_features.get("first_time_borrower", 0)

    if first_timer:
        checks_warnings.append({
            "check":   "CHECK 5 — First Loan Pathway",
            "status":  "  FIRST TIME BORROWER",
            "detail":  "No prior loan history. Score based on savings discipline "
                       "and income stability only. Rider is unknown — not risky.",
            "action":  "Starter loan pathway activated. Max KES 15,000. "
                       "Guarantor required. 3-month maximum term."
        })
        adjustments.append("first_loan_pathway")
    else:
        n_loans = rider_features.get("total_loans_taken", 0)
        checks_passed.append({
            "check":  "CHECK 5 — First Loan Pathway",
            "status": "✅ PASSED",
            "detail": f"Rider has {n_loans} prior loan(s) on record. "
                      f"Full credit history available for scoring."
        })

    # ────────────────────────────────────────────────────────────────────────
    # CHECK 6 — Segment Fairness
    # Hired riders should not be penalised beyond their actual risk
    # ────────────────────────────────────────────────────────────────────────
    bike_owned     = rider_features.get("bike_owned", 0)
    segment_score  = rider_features.get("segment_risk_score", 3)

    if bike_owned == 0:
        checks_warnings.append({
            "check":   "CHECK 6 — Segment Fairness",
            "status":  " HIRED RIDER",
            "detail":  "Rider hires their motorcycle. Daily hire fee creates "
                       "fixed expense regardless of earnings. "
                       "This is factored into NDI calculation — not double counted.",
            "action":  "Hire fee already deducted from NDI. No additional penalty."
        })
    else:
        checks_passed.append({
            "check":  "CHECK 6 — Segment Fairness",
            "status": " PASSED",
            "detail": "Rider owns their motorcycle. "
                      "No hire fee deduction required."
        })

    # ────────────────────────────────────────────────────────────────────────
    # CHECK 7 — Missing Feature Imputation Confirmation
    # Confirm that missing features used neutral defaults
    # ────────────────────────────────────────────────────────────────────────
    checks_passed.append({
        "check":  "CHECK 7 — Missing Feature Imputation",
        "status": " CONFIRMED",
        "detail": "All missing features filled with population medians. "
                  "Absence of data is treated as unknown — not negative."
    })

    # ────────────────────────────────────────────────────────────────────────
    # CHECK 8 — Over-Indebtedness Hard Block
    # Total debt service above 50% of income is a hard stop
    # ────────────────────────────────────────────────────────────────────────
    debt_service       = rider_features.get("debt_service_ratio", 0)
    outstanding_debt   = rider_features.get("total_outstanding_debt", 0)
    requested_monthly  = requested_amount / 3   # assume 3 month term

    # Calculate total debt service ratio including new loan
    total_dsr = debt_service + (requested_monthly / max(monthly_income, 1))

    if total_dsr > 0.50:
        checks_blocks.append({
            "check":  "CHECK 8 — Over-Indebtedness",
            "status": " HARD BLOCK",
            "detail": f"Total debt service ratio with new loan = "
                      f"{total_dsr*100:.1f}%. "
                      f"Exceeds 50% over-indebtedness threshold. "
                      f"Approving this loan would put rider in financial distress.",
            "action": "DECLINE — Rider must reduce existing obligations first. "
                      "Suggest clearing digital loans before reapplying."
        })
    elif total_dsr > 0.35:
        checks_warnings.append({
            "check":   "CHECK 8 — Over-Indebtedness",
            "status":  "  ELEVATED DEBT SERVICE",
            "detail":  f"Total debt service ratio = {total_dsr*100:.1f}%. "
                       f"Above 35% — approaching over-indebtedness threshold.",
            "action":  "Loan amount reduced to keep total DSR below 35%"
        })
    else:
        checks_passed.append({
            "check":  "CHECK 8 — Over-Indebtedness",
            "status": " PASSED",
            "detail": f"Total debt service ratio = {total_dsr*100:.1f}%. "
                      f"Well within safe lending limits."
        })

    # ────────────────────────────────────────────────────────────────────────
    # CHECK 9 — Geographic Fairness
    # Riders from underserved areas should not be penalised for thin data
    # ────────────────────────────────────────────────────────────────────────
    checks_passed.append({
        "check":  "CHECK 9 — Geographic Fairness",
        "status": " CONFIRMED",
        "detail": "Geographic location not used as a scoring factor. "
                  "Thin data in some areas handled by Check 4 separately."
    })

    # ── Determine if any hard blocks exist ───────────────────────────────────
    has_hard_block = len(checks_blocks) > 0

    return {
        "passed":         checks_passed,
        "warnings":       checks_warnings,
        "blocks":         checks_blocks,
        "adjustments":    adjustments,
        "has_hard_block": has_hard_block,
        "data_richness":  richness_score,
    }

print(" All 9 fairness checks defined")

 All 9 fairness checks defined


In [3]:
def print_fairness_report(fairness_result, rider_name):
   
    print()
    print("─" * 55)
    print(f"  FAIRNESS AND RISK CHECKS — {rider_name.upper()}")
    print("─" * 55)

    # Hard blocks first — most important
    if fairness_result["has_hard_block"]:
        print()
        print("  HARD BLOCKS — OVERRIDE DECISION:")
        for block in fairness_result["blocks"]:
            print(f"  {block['status']}: {block['check']}")
            print(f"    {block['detail']}")
            print(f"    Action: {block['action']}")

    # Warnings
    if fairness_result["warnings"]:
        print()
        print("    WARNINGS:")
        for warning in fairness_result["warnings"]:
            print(f"  {warning['status']}: {warning['check']}")
            print(f"    {warning['detail']}")
            if "action" in warning:
                print(f"    Action: {warning['action']}")

    # Passed checks
    if fairness_result["passed"]:
        print()
        print("  CHECKS PASSED:")
        for passed in fairness_result["passed"]:
            print(f"  {passed['status']}: {passed['check']}")
            print(f"    {passed['detail']}")

    # Adjustments summary
    if fairness_result["adjustments"]:
        print()
        print("  ADJUSTMENTS APPLIED:")
        adjustment_labels = {
            "seasonal_adjustment":   "Income volatility penalty waived — rain season",
            "gap_penalty_reduced":   "Income gap penalty reduced — recovery confirmed",
            "thin_data_cap":         "Maximum loan capped — thin data",
            "very_thin_data_cap":    "Maximum loan capped — very thin data",
            "first_loan_pathway":    "First loan pathway activated",
        }
        for adj in fairness_result["adjustments"]:
            print(f"  📋 {adjustment_labels.get(adj, adj)}")

    print("─" * 55)

print(" Fairness report printer defined")

 Fairness report printer defined


In [4]:
def complete_credit_application(rider_features, requested_amount,
                                  requested_term_days, rider_name,
                                  application_month=None):
    """
    Complete credit application processing.
    Runs scoring pipeline + all 9 fairness checks.
    Returns final decision with full audit trail.
    """
    from scoring_pipeline_functions import (
        validate_rider_data,
        prepare_features,
        run_ml_scoring,
        calculate_loan_recommendation,
        print_credit_decision
    )

    # Add loan request features
    monthly_income    = rider_features.get("monthly_income_estimate", 20800)
    estimated_ndi     = rider_features.get("estimated_ndi", 8000)
    requested_monthly = requested_amount / max(requested_term_days / 30, 1)

    rider_features["loan_to_income_ratio"]   = round(
        requested_amount / max(monthly_income, 1), 4)
    rider_features["loan_to_ndi_ratio"]      = round(
        requested_amount / max(estimated_ndi, 1), 4)
    rider_features["repayment_to_ndi_ratio"] = round(
        requested_monthly / max(estimated_ndi, 1), 4)
    rider_features["loan_purpose_code"]      = rider_features.get(
        "loan_purpose_code", 3)

    # Run fairness checks FIRST
    fairness = run_fairness_checks(
        rider_features, {"credit_score": 50},
        requested_amount, application_month
    )

    # If hard block exists — decline immediately
    if fairness["has_hard_block"]:
        print()
        print("═" * 55)
        print(f"  APPLICATION — {rider_name}")
        print("═" * 55)
        print(f"   HARD BLOCK — APPLICATION DECLINED")
        print(f"  Reason: {fairness['blocks'][0]['detail']}")
        print("═" * 55)
        print_fairness_report(fairness, rider_name)
        return {"decision": "HARD_BLOCK", "fairness": fairness}

    # Validate data
    validation = validate_rider_data(rider_features)
    if not validation["is_valid"]:
        print(f"   VALIDATION FAILED: {validation['errors']}")
        return {"decision": "INVALID_DATA", "fairness": fairness}

    # Prepare features and score
    feature_array  = prepare_features(rider_features)
    scoring        = run_ml_scoring(feature_array)

    # Re-run fairness with actual score
    fairness = run_fairness_checks(
        rider_features, scoring,
        requested_amount, application_month
    )

    # Get loan recommendation
    recommendation = calculate_loan_recommendation(
        rider_features, scoring,
        requested_amount, requested_term_days
    )

    # Print complete output
    print_credit_decision(
        {"valid": True, "warnings": [],
         "scoring": scoring, "recommendation": recommendation},
        rider_name, requested_amount, requested_term_days
    )
    print_fairness_report(fairness, rider_name)

    return {
        "decision":       recommendation["decision"],
        "scoring":        scoring,
        "recommendation": recommendation,
        "fairness":       fairness,
    }

print("✅ Complete application function defined")

✅ Complete application function defined


In [5]:
# Since complete_credit_application imports from scoring pipeline
# we test fairness checks directly on our sample riders

print("TESTING FAIRNESS CHECKS ON 3 RIDERS")
print()


john_features = {
    "avg_daily_income":          950.0,
    "total_income_90d":          85500.0,
    "income_volatility_cv":      0.18,
    "income_trend_90d":          0.04,
    "rain_season_dip":           0.22,
    "income_gap_max_days":       2,
    "monthly_income_estimate":   24700.0,
    "estimated_ndi":             12400.0,
    "debt_service_ratio":        0.10,
    "total_outstanding_debt":    0.0,
    "sacco_tenure_months":       14,
    "total_loans_taken":         3,
    "first_time_borrower":       0,
    "ever_defaulted":            0,
    "bike_owned":                1,
    "has_app":                   1,
    "segment_risk_score":        1,
}

john_score = {"credit_score": 69.5}

john_fairness = run_fairness_checks(
    john_features, john_score,
    requested_amount=20000,
    application_month=4    # April — rain season
)
print_fairness_report(john_fairness, "John Kamau")

print()

# ── Peter Otieno — over-indebted stage rider ──────────────────────────────────
peter_features = {
    "avg_daily_income":          620.0,
    "total_income_90d":          55800.0,
    "income_volatility_cv":      0.52,
    "income_trend_90d":         -0.03,
    "rain_season_dip":           0.35,
    "income_gap_max_days":       8,
    "monthly_income_estimate":   16120.0,
    "estimated_ndi":             3000.0,
    "debt_service_ratio":        0.45,
    "total_outstanding_debt":    8500.0,
    "sacco_tenure_months":       0,
    "total_loans_taken":         4,
    "first_time_borrower":       0,
    "ever_defaulted":            1,
    "bike_owned":                0,
    "has_app":                   0,
    "segment_risk_score":        5,
}

peter_score = {"credit_score": 33.0}

peter_fairness = run_fairness_checks(
    peter_features, peter_score,
    requested_amount=15000,
    application_month=7    # July — normal month
)
print_fairness_report(peter_fairness, "Peter Otieno")

print()

# ── James Mwangi — first time borrower ───────────────────────────────────────
james_features = {
    "avg_daily_income":          820.0,
    "total_income_90d":          73800.0,
    "income_volatility_cv":      0.25,
    "income_trend_90d":          0.02,
    "rain_season_dip":           0.18,
    "income_gap_max_days":       3,
    "monthly_income_estimate":   21320.0,
    "estimated_ndi":             9500.0,
    "debt_service_ratio":        0.08,
    "total_outstanding_debt":    0.0,
    "sacco_tenure_months":       0,
    "total_loans_taken":         0,
    "first_time_borrower":       1,
    "ever_defaulted":            0,
    "bike_owned":                1,
    "has_app":                   1,
    "segment_risk_score":        3,
}

james_score = {"credit_score": 52.5}

james_fairness = run_fairness_checks(
    james_features, james_score,
    requested_amount=10000,
    application_month=6    # June — normal month
)
print_fairness_report(james_fairness, "James Mwangi")

TESTING FAIRNESS CHECKS ON 3 RIDERS


───────────────────────────────────────────────────────
  FAIRNESS AND RISK CHECKS — JOHN KAMAU
───────────────────────────────────────────────────────

    WARNINGS:
    RAIN SEASON APPLICATION: CHECK 1 — Seasonal Income
    Application received in April. Income dips during this period are seasonal and consistent with all Nairobi boda-boda riders. Income volatility not penalised this month.
    Action: Income volatility penalty waived for this application
    ELEVATED DEBT SERVICE: CHECK 8 — Over-Indebtedness
    Total debt service ratio = 37.0%. Above 35% — approaching over-indebtedness threshold.
    Action: Loan amount reduced to keep total DSR below 35%

  CHECKS PASSED:
   PASSED: CHECK 2 — Income Gaps
    Longest income gap: 2 days. Short gaps are normal rest or maintenance days.
  CONFIRMED: CHECK 3 — Gender Exclusion
    Gender was not used in any part of the credit score calculation. Compliant with Kenya Data Protection Act 2019.
  ADEQUA

In [6]:
print("PORTFOLIO FAIRNESS DASHBOARD")
print("Monitoring fairness across all riders")
print()

# Get predictions for all riders
X_proba = (
    xgb_model.predict_proba(X)[:, 1] * 0.55 +
    lgb_model.predict_proba(X)[:, 1] * 0.45
)
X_scores  = 100 - (X_proba * 100)
X_preds   = (X_proba >= THRESHOLD).astype(int)

# Add predictions to dataframe
results_df = df.copy()
results_df["credit_score"] = X_scores
results_df["predicted_default"] = X_preds
results_df["approval"] = (X_preds == 0).astype(int)

# Merge with riders for segment info
results_df = results_df.merge(
    riders[["rider_id", "rider_segment",
            "bike_ownership", "gender"]],
    on="rider_id", how="left"
)

print("─" * 55)
print("1. APPROVAL RATE BY RIDER SEGMENT")
print("─" * 55)
segment_stats = (results_df.groupby("rider_segment")
                 .agg(
                     total=("rider_id", "count"),
                     approved=("approval", "sum"),
                     avg_score=("credit_score", "mean"),
                     actual_default_rate=("actual_default", "mean")
                 )
                 .round(3))
segment_stats["approval_rate"] = (
    segment_stats["approved"] / segment_stats["total"]
).round(3)
print(segment_stats.to_string())

print()
print("─" * 55)
print("2. APPROVAL RATE BY BIKE OWNERSHIP")
print("─" * 55)
ownership_stats = (results_df.groupby("bike_ownership")
                   .agg(
                       total=("rider_id", "count"),
                       approved=("approval", "sum"),
                       avg_score=("credit_score", "mean"),
                       actual_default_rate=("actual_default", "mean")
                   )
                   .round(3))
ownership_stats["approval_rate"] = (
    ownership_stats["approved"] / ownership_stats["total"]
).round(3)
print(ownership_stats.to_string())

print()
print("─" * 55)
print("3. GENDER FAIRNESS CHECK")
print("   Gender not used in scoring — verifying score parity")
print("─" * 55)
gender_stats = (results_df.groupby("gender")
                .agg(
                    total=("rider_id", "count"),
                    avg_score=("credit_score", "mean"),
                    approval_rate=("approval", "mean"),
                    actual_default_rate=("actual_default", "mean")
                )
                .round(3))
print(gender_stats.to_string())

# Check gender score gap
if "Male" in gender_stats.index and "Female" in gender_stats.index:
    score_gap = abs(
        gender_stats.loc["Male", "avg_score"] -
        gender_stats.loc["Female", "avg_score"]
    )
    approval_gap = abs(
        gender_stats.loc["Male", "approval_rate"] -
        gender_stats.loc["Female", "approval_rate"]
    )
    print()
    if score_gap <= 5:
        print(f"   Gender score gap: {score_gap:.1f} pts — within acceptable range")
    else:
        print(f"    Gender score gap: {score_gap:.1f} pts — review recommended")

    if approval_gap <= 0.10:
        print(f"   Gender approval gap: {approval_gap:.1%} — within acceptable range")
    else:
        print(f"    Gender approval gap: {approval_gap:.1%} — review recommended")

print()
print("─" * 55)
print("4. SCORE DISTRIBUTION")
print("─" * 55)
bins = [0, 35, 50, 65, 80, 100]
labels = ["Very High Risk", "High Risk", "Medium Risk",
          "Low-Medium Risk", "Low Risk"]
results_df["risk_band"] = pd.cut(
    results_df["credit_score"],
    bins=bins, labels=labels, right=True
)
band_counts = results_df["risk_band"].value_counts().sort_index()
total       = len(results_df)
print()
for band, count in band_counts.items():
    pct = count / total * 100
    bar = "█" * int(pct / 2)
    print(f"  {band:<18} {count:>4} riders ({pct:>5.1f}%)  {bar}")

print()
print("─" * 55)
print("5. FAIRNESS DASHBOARD SUMMARY")
print("─" * 55)

# Calculate fairness indicators
sacco_approval = results_df[
    results_df["rider_segment"] == "SACCO Member"
]["approval"].mean()
stage_approval = results_df[
    results_df["rider_segment"] == "Stage Rider"
]["approval"].mean()
segment_gap    = sacco_approval - stage_approval

owned_approval = results_df[
    results_df["bike_ownership"] == "Owned"
]["approval"].mean()
hired_approval = results_df[
    results_df["bike_ownership"] == "Hired"
]["approval"].mean()
ownership_gap  = owned_approval - hired_approval

indicators = [
    ("Segment approval gap (SACCO vs Stage)",
     segment_gap, 0.40, "higher gap expected — reflects real risk difference"),
    ("Ownership approval gap (Owned vs Hired)",
     ownership_gap, 0.30, "some gap expected — hired riders have higher fixed costs"),
]

for name, gap, threshold, note in indicators:
    status = "✅" if gap <= threshold else " "
    print(f"  {status} {name}: {gap:.1%}")
    print(f"     Note: {note}")
    print()

PORTFOLIO FAIRNESS DASHBOARD
Monitoring fairness across all riders



KeyError: "['rider_segment', 'bike_ownership', 'gender'] not in index"

In [7]:
import pickle

# Save fairness check functions for use in FastAPI backend
fairness_package = {
    "run_fairness_checks":   run_fairness_checks,
    "print_fairness_report": print_fairness_report,
    "RAIN_MONTHS":           RAIN_MONTHS,
    "RAIN_MONTH_NAMES":      RAIN_MONTH_NAMES,
}

with open("models/fairness_checks.pkl", "wb") as f:
    pickle.dump(fairness_package, f)

print("✅ Fairness checks saved")
print("   models/fairness_checks.pkl")
print()
print("=" * 45)
print("  CODE PHASE 6 COMPLETE")
print("  Next: Code Phase 7 — FastAPI Backend")
print("=" * 45)

✅ Fairness checks saved
   models/fairness_checks.pkl

  CODE PHASE 6 COMPLETE
  Next: Code Phase 7 — FastAPI Backend
